In [ ]:
!pip install pyspark pandas numpy

In [ ]:
import os
import time
import numpy as np
import pandas as pd

TOTAL_ROWS = 10_000_000  # 1 Crore rows
CHUNK_SIZE = 1_000_000   # 10 Lakh per file
NUM_CHUNKS = TOTAL_ROWS // CHUNK_SIZE

RAW_DATA_DIR = "/content/data/raw/"
os.makedirs(RAW_DATA_DIR, exist_ok=True)

MERCHANT_CATEGORIES = ['Retail', 'Travel', 'Food', 'Electronics', 'Digital_Services']
LOCATIONS = ['Domestic', 'International']
TXN_TYPES = ['POS', 'Online', 'ATM']

print(f"Generating {TOTAL_ROWS:,} rows in Colab...")
start_time = time.time()

for chunk_id in range(1, NUM_CHUNKS + 1):
    transaction_ids = np.arange((chunk_id - 1) * CHUNK_SIZE, chunk_id * CHUNK_SIZE)
    user_ids = np.random.randint(1000, 500000, size=CHUNK_SIZE)
    amounts = np.round(np.random.lognormal(mean=7.0, sigma=1.5, size=CHUNK_SIZE), 2)
    categories = np.random.choice(MERCHANT_CATEGORIES, size=CHUNK_SIZE)
    locations = np.random.choice(LOCATIONS, size=CHUNK_SIZE, p=[0.85, 0.15])
    txn_types = np.random.choice(TXN_TYPES, size=CHUNK_SIZE)

    is_anomaly = np.zeros(CHUNK_SIZE, dtype=int)
    high_value_mask = amounts > 15000
    risk_loc_mask = locations == 'International'
    risk_type_mask = np.isin(txn_types, ['Online', 'ATM'])

    combined_mask = high_value_mask & risk_loc_mask & risk_type_mask
    anomaly_chances = np.random.random(size=CHUNK_SIZE)
    is_anomaly[combined_mask & (anomaly_chances > 0.2)] = 1

    noise_mask = np.random.random(size=CHUNK_SIZE) < 0.001
    is_anomaly[noise_mask] = 1

    df = pd.DataFrame({
        'transaction_id': transaction_ids, 'user_id': user_ids, 'amount': amounts,
        'merchant_category': categories, 'location': locations,
        'txn_type': txn_types, 'is_anomaly': is_anomaly
    })

    missing_mask = np.random.random(size=CHUNK_SIZE) < 0.02
    df.loc[missing_mask, 'merchant_category'] = np.nan

    file_path = os.path.join(RAW_DATA_DIR, f"transactions_part_{chunk_id:03d}.csv")
    df.to_csv(file_path, index=False)

print(f"✅ Data generated in {time.time() - start_time:.2f} seconds!")

Generating 10,000,000 rows in Colab...
✅ Data generated in 45.13 seconds!


# Big Data Pipeline: Ingestion and Preprocessing

In [ ]:
import os
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, isnull, sum as spark_sum
from pyspark.ml.feature import StringIndexer, VectorAssembler

In [ ]:
#force local network binding to bypass windows firewall hangs
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"

def create_spark_session(app_name: str) -> SparkSession:
    """
    creates and configures a spark session optimized for local development.

    args:
        app_name (str): the name of the spark application.

    returns:
        sparksession: the configured spark session object.
    """
    #reduced memory allocation to prevent windows memory thrashing
    #added explicit localhost bind address for driver
    spark = SparkSession.builder \
        .appName(app_name) \
        .config("spark.driver.memory", "4g") \
        .config("spark.executor.memory", "4g") \
        .config("spark.driver.bindAddress", "127.0.0.1") \
        .config("spark.sql.shuffle.partitions", "100") \
        .master("local[*]") \
        .getOrCreate()

    return spark

#instantiate the session
spark = create_spark_session("seshat_bigdata_ingestion")
print("spark session successfully initialized!")

spark session successfully initialized!


## Data Ingestion
Loading the distributed csv files from the raw data lake directory into a unified PySpark DataFrame.

In [ ]:
def ingest_raw_data(spark_session: SparkSession, data_path: str):
    """
    reads distributed csv files from the specified directory into a spark dataframe.

    args:
        spark_session (sparksession): the active spark session.
        data_path (str): the relative or absolute path to the raw data folder.

    returns:
        dataframe: the loaded pyspark dataframe.
    """
    #read all parts from the raw data directory inferring schema automatically
    df = spark_session.read.csv(data_path, header=True, inferSchema=True)
    return df

In [ ]:
#define path and load data
raw_data_path = RAW_DATA_DIR
df_raw = ingest_raw_data(spark, raw_data_path)

#display schema to verify column data types
df_raw.printSchema()

root
 |-- transaction_id: integer (nullable = true)
 |-- user_id: integer (nullable = true)
 |-- amount: double (nullable = true)
 |-- merchant_category: string (nullable = true)
 |-- location: string (nullable = true)
 |-- txn_type: string (nullable = true)
 |-- is_anomaly: integer (nullable = true)



## Data Cleaning and Missing Value Handling
Identifying null values within the dataset and applying imputation strategies to ensure data integrity before modeling.

In [ ]:
def handle_missing_values(df):
    """
    identifies and imputes missing values in the dataframe.
    fills categorical nulls with a default string indicator.

    args:
        df (dataframe): the raw pyspark dataframe.

    returns:
        dataframe: the cleaned dataframe with imputed values.
    """
    #fill missing categorical values with a static placeholder
    df_cleaned = df.fillna({"merchant_category": "unknown"})
    return df_cleaned


In [ ]:
#apply cleaning function
df_clean = handle_missing_values(df_raw)

## Feature Engineering
Transforming categorical string variables into numerical formats necessary for distributed machine learning algorithms.

In [ ]:
def perform_feature_engineering(df):
    """
    transforms categorical string columns into numerical indices using stringindexer.

    args:
        df (dataframe): the cleaned pyspark dataframe.

    returns:
        dataframe: the engineered dataframe ready for machine learning.
    """
    #define categorical columns to index for the model
    indexers = [
        StringIndexer(inputCol="merchant_category", outputCol="merchant_category_idx"),
        StringIndexer(inputCol="location", outputCol="location_idx"),
        StringIndexer(inputCol="txn_type", outputCol="txn_type_idx")
    ]

    #apply string indexers iteratively across the dataframe
    df_transformed = df
    for indexer in indexers:
        df_transformed = indexer.fit(df_transformed).transform(df_transformed)

    return df_transformed

In [ ]:
#execute feature engineering
df_engineered = perform_feature_engineering(df_clean)

#verify transformed columns
df_engineered.printSchema()

root
 |-- transaction_id: integer (nullable = true)
 |-- user_id: integer (nullable = true)
 |-- amount: double (nullable = true)
 |-- merchant_category: string (nullable = false)
 |-- location: string (nullable = true)
 |-- txn_type: string (nullable = true)
 |-- is_anomaly: integer (nullable = true)
 |-- merchant_category_idx: double (nullable = false)
 |-- location_idx: double (nullable = false)
 |-- txn_type_idx: double (nullable = false)



## Export Processed Data
Saving the fully engineered DataFrame into a highly compressed columnar format (Parquet) for efficient loading in the subsequent modeling phase.

In [ ]:
def save_processed_data(df, output_path: str):
    """
    saves the engineered dataframe to the specified path in parquet format.

    args:
        df (dataframe): the fully processed pyspark dataframe.
        output_path (str): the destination path for the parquet files.
    """
    #write dataframe to parquet format using overwrite mode to prevent duplication errors
    df.write.mode("overwrite").parquet(output_path)

PROCESSED_DATA_DIR = "/content/data/processed/"
os.makedirs(PROCESSED_DATA_DIR, exist_ok=True)

#define output destination and save the processed data lake
processed_data_path = "/content/data/processed/transactions.parquet"
save_processed_data(df_engineered, processed_data_path)

# ML Modeling and Seshat Knowledge Extraction

In [ ]:
from pyspark.sql import SparkSession
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
import pandas as pd

def initialize_ml_spark_session(app_name: str) -> SparkSession:
    """
    initializes a spark session specifically for distributed machine learning tasks.

    args:
        app_name (str): name of the spark application.

    returns:
        sparksession: active spark session object.
    """
    #allocate memory for model training and shuffle operations
    spark = SparkSession.builder \
        .appName(app_name) \
        .master("local[*]") \
        .getOrCreate()

    return spark

#start the session
spark = initialize_ml_spark_session("seshat_ml_modeling")

## Load Processed Data
Loading the highly compressed Parquet data that was cleaned and engineered in the previous ingestion phase.

In [ ]:
def load_parquet_data(spark_session: SparkSession, file_path: str):
    """
    loads the engineered parquet data into a dataframe.

    args:
        spark_session (sparksession): active spark session.
        file_path (str): path to the processed parquet directory.

    returns:
        dataframe: pyspark dataframe containing cleaned data.
    """
    #read parquet files into memory
    df = spark_session.read.parquet(file_path)
    return df

#define path based on the colab environment established earlier
processed_path = "/content/data/processed/transactions.parquet"
df_processed = load_parquet_data(spark, processed_path)

## Feature Assembly
Spark MLlib requires all input features to be combined into a single vector column before training.

In [ ]:
def assemble_feature_vector(df):
    """
    combines individual numerical and indexed categorical columns into a single feature vector.

    args:
        df (dataframe): the input dataframe with engineered columns.

    returns:
        dataframe: dataframe containing the new combined vector column.
    """
    #define which columns represent the independent variables
    feature_columns = [
        "amount",
        "merchant_category_idx",
        "location_idx",
        "txn_type_idx"
    ]

    #initialize the assembler pointing to the target output column
    assembler = VectorAssembler(inputCols=feature_columns, outputCol="features")

    #transform the dataframe to include the feature vector
    df_assembled = assembler.transform(df)
    return df_assembled, feature_columns

#apply assembly and retrieve feature list for later knowledge extraction
df_model_ready, feature_list = assemble_feature_vector(df_processed)

## Distributed Model Training and Evaluation
Splitting the data and training a Random Forest Classifier. This model is chosen because its tree-based structure allows for easy extraction of feature importance, which is critical for the Seshat AI reasoning approach.

In [ ]:
def train_and_evaluate_model(df):
    """
    splits data trains a random forest model and evaluates its performance metrics.

    args:
        df (dataframe): dataframe containing the feature vector and label.

    returns:
        randomforestclassificationmodel: the trained machine learning model.
    """
    #divide the dataset randomly for training and validation purposes
    train_data, test_data = df.randomSplit([0.8, 0.2], seed=42)

    #configure the random forest classifier pointing to our specific label
    rf = RandomForestClassifier(featuresCol="features", labelCol="is_anomaly", numTrees=20)

    #execute distributed training across the cluster
    rf_model = rf.fit(train_data)

    #generate predictions on the unseen test dataset
    predictions = rf_model.transform(test_data)

    #evaluate the accuracy of the predictions
    evaluator = MulticlassClassificationEvaluator(
        labelCol="is_anomaly", predictionCol="prediction", metricName="accuracy"
    )
    accuracy = evaluator.evaluate(predictions)

    print(f"model evaluation accuracy: {accuracy:.4f}")

    return rf_model

#execute the training pipeline
trained_model = train_and_evaluate_model(df_model_ready)

model evaluation accuracy: 0.9980


## Seshat AI Knowledge Extraction
Extracting the mathematical feature importance from the model to discover which patterns actually drive the anomalies. This fulfills the "pattern discovery" requirement and will be passed to the LLM reasoning layer later.

In [ ]:
def extract_seshat_insights(model, features):
    """
    extracts feature importance scores from the model and maps them to feature names.
    this forms the foundational knowledge base for the seshat reasoning engine.

    args:
        model (randomforestclassificationmodel): the trained model.
        features (list): list of string names for the input features.

    returns:
        dataframe: pandas dataframe containing features and their importance weights.
    """
    #retrieve the raw importance array from the trained model
    importances = model.featureImportances.toArray()

    #map the numerical scores back to their human readable feature names
    importance_data = []
    for feature, score in zip(features, importances):
        importance_data.append({"feature": feature, "importance_score": float(score)})

    #convert to a pandas dataframe for easy viewing and subsequent export
    importance_df = pd.DataFrame(importance_data).sort_values(by="importance_score", ascending=False)

    print("seshat extracted knowledge rules:")
    print(importance_df.to_string(index=False))

    return importance_df

#extract and display the discovered patterns
knowledge_base = extract_seshat_insights(trained_model, feature_list)

seshat extracted knowledge rules:
              feature  importance_score
         location_idx          0.550085
               amount          0.331031
         txn_type_idx          0.118878
merchant_category_idx          0.000005


## Model Serialization
Saving the trained Spark MLlib model to disk so it can be loaded by the insight generation module without needing to retrain.

In [ ]:
def save_model(model, output_path: str):
    """
    saves the trained model artifact to the specified directory.

    args:
        model (randomforestclassificationmodel): the trained model to save.
        output_path (str): the destination directory path.
    """
    #write the model to disk overwriting any existing model in that directory
    model.write().overwrite().save(output_path)
    print(f"model successfully saved to {output_path}")

#define the output path and save
model_output_path = "/content/models/seshat_anomaly_model"
save_model(trained_model, model_output_path)

model successfully saved to /content/models/seshat_anomaly_model


In [ ]:
import shutil
from google.colab import files

def download_spark_model(model_dir: str, output_zip_name: str):
    """
    compresses the distributed spark model directory into a zip file and triggers a browser download.

    args:
        model_dir (str): the path to the saved spark model directory.
        output_zip_name (str): the desired name for the output zip archive without the extension.
    """
    #compress the directory into a single zip archive
    shutil.make_archive(output_zip_name, 'zip', model_dir)

    #trigger the download to the local machine
    zip_file_path = f"{output_zip_name}.zip"
    files.download(zip_file_path)

#define paths and execute the download
colab_model_path = "/content/models/seshat_anomaly_model"
zip_filename = "/content/seshat_anomaly_model"

download_spark_model(colab_model_path, zip_filename)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>